1단계: PDF 텍스트 추출
- 라이브러리 설치: !pip install pdfplumber

In [2]:
import pdfplumber # PDF 라리브러리 import

# PDF 파일 열기
with pdfplumber.open(path_or_fp="sample_korean.pdf") as pdf:
    text = ""
    for page in pdf.pages:
        text += page.extract_text() + "\n"

print(text[:300]) # 앞부분만 확인

제목: 인공지능과 벡터 데이터베이스
인공지능은 현대 사회에서 중요한 역할을 하고 있습니다.
특히 대규모 언어 모델(LLM)은 방대한 텍스트 데이터를 학습하여
사람과 유사한 방식으로 질문에 답변할 수 있습니다.
벡터 데이터베이스는 이러한 LLM과 함께 사용될 때 강력한 도구가 됩니다.
문서를 벡터로 변환하여 저장하면, 질문과 가장 유사한 문서를 빠르게 검색할 수 있습니다.
Qdrant는 대표적인 벡터 데이터베이스로,
고성능 검색과 인덱싱을 지원하며 RAG 시스템 구축에 널리 활용됩니다.



2단계: 임베딩 생성
- 라이브러리 설치: !pip install sentence-transformers

In [3]:
from sentence_transformers import SentenceTransformer # 임베딩 라이브러리 import

# 모델 로드: 다국어 임베딩 모델(한국어 포함, 384차원)
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# 문단 단위로 나누기
chunks = text.split("\n")
embeddings = [model.encode(chunk).tolist() for chunk in chunks if chunk.strip()]

print(len(embeddings), "개의 벡터 생성 완료")

8 개의 벡터 생성 완료


3단계: Qdrant 컬렉션 생성
- 라이브러리 설치: !pip install qdrant-client

In [4]:
from qdrant_client import QdrantClient # Qdrant 클라이언트 import
from qdrant_client.http import models # Qdrant 클라이언트에서 벡터 파라미터/PointStruct 구조체 import

# Qdrant 서버 연결
client = QdrantClient(host="localhost", port=6333)

collection_name = "documents"

# 컬렉션 존재 여부 확인
try:
    client.get_collection(collection_name=collection_name)
    print(f" '{collection_name}' 컬렉션이 이미 존재합니다.")
except Exception:
    # 컬렉션이 없으면 새로 컬렉션 생성
    client.create_collection(
        collection_name=collection_name,
        vectors_config=models.VectorParams(size=len(embeddings[0]), distance=models.Distance.COSINE),
    )
    print(f" '{collection_name}' 컬렉션이 새로 생성되었습니다.")

# 컬렉션 생성 : 항상 새로 컬랙션 생성(덮어쓰기)
# client.recreate_collection(
#     collection_name="documents",
#     vectors_config=models.VectorParams(size=len(embeddings[0]), distance=models.Distance.COSINE),
# )

 'documents' 컬렉션이 이미 존재합니다.


4단계: Qdrant 인덱싱
- Upsert 인덱싱 작업

In [5]:
points = [
    models.PointStruct(
        id=i,
        vector=embeddings[i],
        payload={"text": chunks[i]}
    )
    for i in range(len(embeddings))
]

client.upsert(collection_name="documents", points=points)

print("인덱싱 완료")

인덱싱 완료


5단계: 검색 결과 확인

In [ ]:
query = "벡터 데이터베이스란 무엇인가?"
# 임베딩: 벡터 변환
query_vector = model.encode(query).tolist()

# 검색 결과
results = client.query_points(
    collection_name="documents",
    query=query_vector,
    limit=3
)

for r in results.points:
    print(r.payload["text"], "=> score:", r.score)

제목: 인공지능과 벡터 데이터베이스 => score: 0.7779458
벡터 데이터베이스는 이러한 LLM과 함께 사용될 때 강력한 도구가 됩니다. => score: 0.7101351
Qdrant는 대표적인 벡터 데이터베이스로, => score: 0.6912204
